# Предобработка датасетов для ЛР 1 по NLP

Датасеты:
1. neo_task.csv - классификация опасности астероидов (hazardous)
2. winequality-red.csv + winequality-white.csv - регрессия качества вина (quality)
3. Текстовый датасет для Word2Vec (отсутствует)

Результат: 2 обработанных файла для PyTorch

## 1. Настройка окружения

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('datasets')
PROCESSED_DIR = Path('processed')
PROCESSED_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', None)

## 2. neo_task.csv (классификация)

In [2]:
neo = pd.read_csv(DATA_DIR / 'neo_task.csv')
print(f"Размер: {neo.shape}")
print(neo.isna().sum())

Размер: (90836, 8)
id                    570
name                   46
est_diameter_min        0
est_diameter_max      457
relative_velocity     557
miss_distance           0
absolute_magnitude    684
hazardous               0
dtype: int64


In [3]:
# Заполнение id по name
neo['id'] = pd.to_numeric(neo['id'], errors='coerce')
names_with_id = neo[neo['id'].notna() & neo['name'].notna()]
name_to_id = names_with_id.groupby('name')['id'].first().to_dict()
mask = neo['id'].isna() & neo['name'].isin(name_to_id)
neo.loc[mask, 'id'] = neo.loc[mask, 'name'].map(name_to_id)

# Генерация новых id
missing_name = neo.loc[neo['id'].isna(), 'name'].dropna().unique()
new_names = [n for n in missing_name if n not in name_to_id]
if new_names:
    current_max = neo['id'].max()
    if pd.isna(current_max):
        current_max = 0
    new_ids = range(int(current_max) + 1, int(current_max) + 1 + len(new_names))
    new_name_to_id = dict(zip(new_names, new_ids))
    mask_new = neo['id'].isna() & neo['name'].isin(new_names)
    neo.loc[mask_new, 'id'] = neo.loc[mask_new, 'name'].map(new_name_to_id)

neo = neo.dropna(subset=['id'])
print(f"После обработки id: {neo.shape}")

После обработки id: (90836, 8)


In [4]:
# Заполнение name по id
mask = neo['name'].isna()
neo.loc[mask, 'name'] = 'asteroid_' + neo.loc[mask, 'id'].astype(int).astype(str)
print(f"Пропусков в name: {neo['name'].isna().sum()}")

Пропусков в name: 0


In [5]:
# Заполнение числовых столбцов по id
for col in ['est_diameter_max', 'relative_velocity', 'absolute_magnitude']:
    valid = neo[neo[col].notna()]
    id_to_val = valid.groupby('id')[col].first().to_dict()
    mask = neo[col].isna() & neo['id'].isin(id_to_val)
    neo.loc[mask, col] = neo.loc[mask, 'id'].map(id_to_val)
    print(f"{col}: заполнено, осталось {neo[col].isna().sum()}")

est_diameter_max: заполнено, осталось 55
relative_velocity: заполнено, осталось 69
absolute_magnitude: заполнено, осталось 85


In [6]:
neo = neo.dropna(subset=['est_diameter_max', 'relative_velocity', 'absolute_magnitude'])
print(f"После удаления: {neo.shape}")

После удаления: (90627, 8)


In [7]:
neo = neo.dropna(subset=['hazardous'])
neo['hazardous'] = neo['hazardous'].map({False: 0, True: 1}).astype(int)
print(neo['hazardous'].value_counts())

hazardous
0    81804
1     8823
Name: count, dtype: int64


In [8]:
neo['id'] = neo['id'].astype(int)
float_cols = ['est_diameter_min', 'est_diameter_max', 'relative_velocity', 'miss_distance', 'absolute_magnitude']
neo[float_cols] = neo[float_cols].astype('float32')
print("Типы оптимизированы")

Типы оптимизированы


In [9]:
neo.to_csv(PROCESSED_DIR / 'neo_task_processed.csv', index=False)
print("neo_task сохранен")

neo_task сохранен


## 3. winequality (регрессия)

In [10]:
red = pd.read_csv(DATA_DIR / 'winequality-red.csv', sep=';')
white = pd.read_csv(DATA_DIR / 'winequality-white.csv', sep=';')
print(f"Красное: {red.shape}, Белое: {white.shape}")

Красное: (1599, 12), Белое: (4898, 12)


In [11]:
red['wine_type'] = 'red'
white['wine_type'] = 'white'
wine = pd.concat([red, white], ignore_index=True)

duplicates = wine.duplicated().sum()
if duplicates > 0:
    wine = wine.drop_duplicates()
    print(f"Удалено {duplicates} дубликатов")

print(f"Объединённый: {wine.shape}")

Удалено 1177 дубликатов
Объединённый: (5320, 13)


In [12]:
num_cols = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
            'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
            'density', 'pH', 'sulphates', 'alcohol', 'quality']
wine[num_cols] = wine[num_cols].astype('float32')
wine['wine_type'] = wine['wine_type'].map({'red': 0, 'white': 1}).astype(int)
print("Типы оптимизированы")

Типы оптимизированы


In [13]:
wine.to_csv(PROCESSED_DIR / 'winequality_processed.csv', index=False)
print("winequality сохранён")

winequality сохранён


In [14]:
print(f"neo: {neo.shape}")
print(f"wine: {wine.shape}")
print(f"\nСохранено в: {PROCESSED_DIR.absolute()}")

neo: (90627, 8)
wine: (5320, 13)

Сохранено в: C:\Users\trape\OneDrive\Desktop\Studies 2026\NLP\solution\lab1\processed
